<a href="https://colab.research.google.com/github/HB0918/NLPforET/blob/main/Past_Tense_of_Irregular_Verbs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
!pip install gradio

import gradio as gr
import random
import time

# 🔹 불규칙 동사 30개
word_list = [
    ("go", "went"), ("eat", "ate"), ("see", "saw"),
    ("come", "came"), ("take", "took"), ("make", "made"),
    ("get", "got"), ("give", "gave"), ("run", "ran"),
    ("have", "had"), ("do", "did"), ("say", "said"),
    ("find", "found"), ("tell", "told"), ("be", "was/were"),
    ("begin", "began"), ("drink", "drank"), ("write", "wrote"),
    ("read", "read"), ("sing", "sang"), ("sit", "sat"),
    ("stand", "stood"), ("lose", "lost"), ("meet", "met"),
    ("pay", "paid"), ("bring", "brought"), ("buy", "bought"),
    ("think", "thought"), ("teach", "taught"), ("catch", "caught")
]

class State:
    def __init__(self):
        self.questions = []
        self.answers = []
        self.index = 0
        self.correct = 0
        self.start_time = 0

state = State()

# 🔹 문제 생성
def generate_questions(mode, num_words):
    selected = random.sample(word_list, int(num_words))
    q, a = [], []

    if mode == "Speed":
        for w, p in selected:
            q.append(f"{w} → ?")
            a.append(p)
    else:
        for w, p in selected:
            q.extend([
                f"{w} → ?",
                f"Yesterday I ___ ({w})",
                f"{w}의 과거형은?",
                f"{w} → ______",
                f"Write the past form of '{w}'"
            ])
            a.extend([p]*5)

    return q, a

# 🔹 시작
def start(mode, num_words):
    state.questions, state.answers = generate_questions(mode, num_words)
    state.index = 0
    state.correct = 0
    state.start_time = time.time()

    return (
        gr.update(visible=False),
        gr.update(visible=True),
        gr.update(visible=False),
        f"<div class='question'>{state.questions[0]}</div>",
        f"<div class='progress'>📊 1 / {len(state.questions)}</div>",
        "",
        "",
        gr.update(visible=True),
        gr.update(visible=False)
    )

# 🔹 정답 확인
def check_answer(user_input):
    correct = state.answers[state.index]

    if user_input.strip().lower() == correct.lower():
        state.correct += 1
        feedback = "<div class='correct'>✅ 정답!</div>"
    else:
        feedback = f"<div class='wrong'>❌ 오답! 정답: {correct}</div>"

    return (
        feedback,
        gr.update(visible=False),
        gr.update(visible=True)
    )

# 🔹 다음 문제
def next_question():
    state.index += 1

    if state.index >= len(state.questions):
        total_time = round(time.time() - state.start_time, 2)
        acc = round(state.correct / len(state.questions) * 100, 1)

        result_html = f"""
        <div class="result-card">
            <div class="result-title">🎉 학습이 끝났습니다!</div>
            <div class="result-score">정답률: {acc}%</div>
            <div class="result-time">총 학습 시간: {total_time}초</div>
        </div>
        """

        return (
            gr.update(visible=False),
            gr.update(visible=True),
            "",
            "",
            "",
            "",
            result_html,
            gr.update(visible=False),
            gr.update(visible=False)
        )

    return (
        gr.update(visible=True),
        gr.update(visible=False),
        f"<div class='question'>{state.questions[state.index]}</div>",
        f"<div class='progress'>📊 {state.index+1} / {len(state.questions)}</div>",
        "",
        "",  # ✔ answer 초기화 (중요)
        "",
        gr.update(visible=True),
        gr.update(visible=False)
    )

# 🔹 다시 시작
def restart():
    return (
        gr.update(visible=True),
        gr.update(visible=False),
        gr.update(visible=False),
        "", "", "", "",
        gr.update(visible=True),
        gr.update(visible=False)
    )

# 🔹 UI
with gr.Blocks(css="""
body {background-color:#f5f7fb;}

/* 문제 */
.question {
    font-size:48px;
    font-weight:bold;
    text-align:center;
    padding:30px;
}

/* 진행률 */
.progress {
    font-size:24px;
    text-align:center;
    margin-bottom:20px;
}

/* 정답/오답 */
.correct {font-size:28px; color:#4CAF50; text-align:center;}
.wrong {font-size:28px; color:#E53935; text-align:center;}

/* 입력창 */
textarea {
    font-size:26px !important;
}

/* 버튼 */
button {
    font-size:24px !important;
    padding:14px !important;
}

/* 결과 */
.result-card {
    background:white;
    padding:50px;
    border-radius:20px;
    text-align:center;
}
.result-title {font-size:44px; font-weight:bold;}
.result-score {font-size:34px; color:#4CAF50;}
.result-time {font-size:30px;}
""") as app:

    with gr.Column(visible=True) as start_screen:
        gr.Markdown("# 📘 Past Tense of Irregular Verbs 📘")
        mode = gr.Radio(["Speed", "Daily"], label="모드 선택")
        num = gr.Slider(1, 30, step=1, value=5, label="단어 개수")
        start_btn = gr.Button("🚀 시작")

    with gr.Column(visible=False) as quiz_screen:
        question = gr.Markdown()
        progress = gr.Markdown()
        answer = gr.Textbox(placeholder="답 입력")
        feedback = gr.Markdown()

        submit_btn = gr.Button("제출")
        next_btn = gr.Button("➡️ 다음", visible=False)

    with gr.Column(visible=False) as result_screen:
        result_text = gr.Markdown()
        restart_btn = gr.Button("🔄 다시 시작")

    start_btn.click(start,[mode, num],
        [start_screen, quiz_screen, result_screen,
         question, progress, feedback, answer,
         submit_btn, next_btn])

    submit_btn.click(check_answer, answer,
        [feedback, submit_btn, next_btn])

    next_btn.click(next_question,[],
        [quiz_screen, result_screen,
         question, progress, feedback, answer,
         result_text, submit_btn, next_btn])

    restart_btn.click(restart,[],
        [start_screen, quiz_screen, result_screen,
         question, progress, feedback, answer,
         submit_btn, next_btn])

app.launch()

/tmp/ipykernel_15182/1262368555.py:140: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css="""


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c2335d9f13c702eed7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
